# 究竟什么是 API？

> API 这个词在 AI、Agent、天气、地图等场景中到处出现。理解它最快的方式，不是先背定义，而是亲自调用几个真正的 API。

从这一节开始进入后端部分。文字实验室中“开始分析”按钮还没有真正计算，拼音和情感分数也是假数据。要让前端把文字交给后端处理，再把结果拿回来，前后端之间靠的就是 API。

## 从一个问题说起

有人听说 Claude 或其他大模型很厉害，找到了提供 API 的渠道，却不知道 API 是什么意思。

API（Application Programming Interface，应用程序编程接口）可以先理解成：一个程序对外公开的固定入口。调用方按照它规定的方式发请求，就能使用它提供的能力，而不必知道内部具体是怎么实现的。

API 不只用于大模型：

- 有的 API 查询天气、地图或公网 IP。
- 有的 API 让程序获得大模型回复。
- 有的 API 执行计算、处理图片或读写数据。
- 我们自己的后端，也会通过 API 把拼音和情感分数提供给前端。

## 第一个例子：查询自己的公网 IP

ipify 是一个免费开放的网站服务，它只做一件事：返回当前请求者的公网 IP。它既提供网页，也提供给程序调用的 API。

### 使用 curl 发请求

curl 是终端里的网络请求工具。浏览器可以访问网址，curl 也可以在命令行中访问网址，并把服务器返回的内容直接打印出来，特别适合测试 API。macOS 和大多数 Linux 系统一般自带 curl。

```bash
curl 'https://api.ipify.org?format=json'
```

网址使用引号括起来更稳妥，因为其中的问号等符号可能被终端特殊处理。返回结果大致是：

```json
{ "ip": "114.86.123.45" }
```

实际返回的 IP 会是你自己的公网 IP。你没有打开网页，只通过终端问到了一个服务的数据，这就叫调用 ipify 的 API。

同一个地址也可以直接粘贴到浏览器地址栏中打开。这说明浏览器访问网址，本质上也是向服务器发请求、接收响应。Windows PowerShell 中若要明确使用 curl 程序，可以写 curl.exe。

## 第二个例子：调用 DeepSeek API

这次不是查询一份固定数据，而是让大模型根据输入返回一句话。服务方需要知道“是谁在调用”，以便统计用量并防止滥用，所以需要 API Key。

### 准备 API Key

1. 注册或登录 DeepSeek 开放平台：https://platform.deepseek.com/。
2. 进入 API keys 页面，创建一个 key。
3. 得到一串通常以 sk- 开头的字符串，并立即保存，因为完整内容通常只显示一次。
4. API 按用量计费，调用前按平台要求准备余额。
5. 参数和模型名称以官方文档为准：https://api-docs.deepseek.com/zh-cn/。

API Key 是密码，不要把它写进公开 Notebook、提交到 GitHub，或发到聊天群。建议放入环境变量中，让命令读取环境变量。

### 用 curl 发送对话请求

下面是课程页面中的请求结构。示例中的模型名称和参数可能会随官方平台更新，实际使用时以官方文档为准。

```bash
curl https://api.deepseek.com/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer ${DEEPSEEK_API_KEY}" \
  -d '{
        "model": "deepseek-v4-pro",
        "messages": [
          {"role": "system", "content": "You are a helpful assistant."},
          {"role": "user", "content": "你好，请用一句话介绍你自己"}
        ],
        "thinking": {"type": "enabled"},
        "reasoning_effort": "high",
        "stream": false
      }'
```

在 Linux 或 macOS 终端中，可以先设置当前终端会话的环境变量：

```bash
export DEEPSEEK_API_KEY="sk-你的密钥"
```

然后将命令中的占位符替换为环境变量值，或按官方文档使用环境变量形式执行。不要把真实密钥替换进命令后再截图或提交到仓库。

## 读懂请求和响应

请求中的几个部分各有作用：

- URL：https://api.deepseek.com/chat/completions，表示要访问的接口地址。
- Content-Type: application/json：告诉服务器请求正文采用 JSON 格式。
- Authorization: Bearer ...：携带身份凭证。
- -d：提交请求正文。
- model：指定使用的模型。
- messages：传给模型的对话消息。
- stream: false：要求一次性返回完整结果，而不是分段流式返回。

把用户消息从：

```json
{"role": "user", "content": "Hello!"}
```

改成：

```json
{"role": "user", "content": "你好，请用一句话介绍你自己"}
```

返回内容删掉次要字段后，大致是：

```json
{
  "choices": [
    {
      "message": {
        "role": "assistant",
        "content": "你好！我是 DeepSeek，一个由深度求索打造的 AI 助手，很高兴为你服务。"
      }
    }
  ]
}
```

真正的回答位于 choices → message → content。这次没有打开网页，而是在终端里让大模型返回了一句话，这就是调用 DeepSeek API。

## API 是设计给计算机程序用的

人可以直接访问 ipify.org 网页，也可以使用 DeepSeek 网页或手机 App。但程序不能每次都打开浏览器、寻找按钮、复制结果。

因此 API 专门为程序调用设计：

1. 调用方向固定 URL 发起请求。
2. 按接口文档提供必要的信息。
3. 服务方在内部处理请求。
4. 服务方返回结构化响应，通常是 JSON。

这种“把自己的能力通过固定入口持续提供给其他程序调用”的方式，是软件世界的通用做法。调用方不必知道对方内部使用了哪种语言或框架，只要遵守接口约定即可。

## API 的格式标准

ipify 和 DeepSeek 的功能完全不同，但调用过程很相似：

```text
请求 URL
  → 按要求携带参数、请求头或请求正文
  → 对方处理
  → 返回响应，通常是 JSON
```

两次请求使用了不同的 HTTP 方法：

- GET：获取数据。查询公网 IP 使用的是 GET。
- POST：提交数据并请求处理。对话 API 需要提交 messages，因此使用 POST。

API 还会用到 PUT、DELETE 等方法：

- PUT 常用于整体更新资源。
- DELETE 常用于删除资源。

方法的具体含义以接口文档为准。只要请求和响应遵守公开标准，调用方和提供方就可以使用不同的编程语言实现。例如，DeepSeek 的内部语言不影响我们用 curl、Python 或 JavaScript 调用它。

## 为什么 AI 时代到处都是 API

许多 AI 产品的核心流程，本质上是调用大模型 API：

1. 把用户输入包装成请求。
2. 发给模型服务。
3. 取回模型回答。
4. 再包装成界面显示给用户。

AI Agent 看起来能做很多事，是因为它在不断调用工具：

- 查询天气、快递和网页。
- 向群聊或其他服务发送消息。
- 调用另一个大模型。
- 在电脑上执行命令、读取和写入文件。

联网服务大多通过 API 调用；本地命令和文件操作虽然不一定是网络 API，但同样遵循“按照固定接口使用现成能力”的思想。

所以理解 API，就揭开了 AI 工具和 Agent 工作方式的一大部分。

## 回到我们的网站：前端调后端也是调 API

现在的网站只有前端，负责浏览器里看得见的展示和交互。文字实验室要根据用户输入计算拼音和情感分数，这部分工作适合交给后端。

未来的调用流程是：

```text
用户在前端输入文字
  → 前端向我们自己的 API 发请求
  → 后端接收文字并计算
  → 后端返回包含拼音和情感分数的 JSON
  → 前端根据 JSON 更新结果区
```

这和刚才调用 DeepSeek 是同一件事，只是接口从 DeepSeek 的 /chat/completions 换成了我们自己编写的接口。

下一步要给网站补上后端，并实现这个 API。

## “后端”到底是什么

- 前端：运行在用户浏览器中，负责看得见的展示和交互。
- 后端：持续运行在服务器上，等待接收请求，负责看不见的计算、处理和数据存储。
- API：前端调用后端能力时使用的公开入口。

ipify 和 DeepSeek 背后都有一个一直运行、等待请求的服务程序。我们的后端规模会小很多，但工作方式相同：接收请求、处理数据、返回响应。

## 后端可以用很多种语言写

提供 API 的后端可以使用 JavaScript（Node.js）、Python、Go、Java、PHP、Ruby、C# 等语言。API 的概念与实现语言无关：

- 调用方不需要知道服务内部使用 Python 还是 C。
- 提供方也不关心调用方使用 curl、Python 还是 JavaScript。
- 双方只要遵守相同的 HTTP 和数据格式约定即可。

本课程选择 Python，主要有两个原因：

1. Python 生态成熟，特别适合算法、数据分析和 AI 模型服务，正好能支持文字实验室的拼音和情感分数。
2. Python 语法相对友好，适合初学者入门。

要记住：Python 只是实现 API 的一种选择，API 本身和具体语言无关。

## 这一节应该带走什么

- API 是程序对外公开的固定入口；按规定发请求，就能使用它的能力。
- 一次 API 调用通常是：请求 → 对方处理 → 响应，响应常见格式是 JSON。
- 你通过 curl 调用了 ipify 和 DeepSeek 两个真实 API。
- GET 常用于取数据，POST 常用于提交内容并请求处理，还有 PUT、DELETE 等方法。
- AI 产品和 Agent 大量依赖 API 来调用现成能力。
- 我们的网站接下来会增加后端，前端通过自己编写的 API 获取拼音和情感分数。
- 后端可以用很多语言实现，本课程选择 Python，但 API 概念与语言无关。

下一节安装 Python 并运行第一个 Python 程序，为亲手写出自己的 API 做准备。